# Assignment Beta, Question 4: LLM sentiment evaluation
Agentic AI (AI4015), Fall 2026. Every number is computed from the confusion matrices and token data in the quues


In [1]:
import math
import numpy as np
import pandas as pd
from collections import Counter

labels = ["Positive", "Negative", "Neutral"]
S1 = np.array([[100, 5, 15],
               [  8, 80, 12],
               [ 20, 10, 50]])
S2 = np.array([[106, 4, 10],
               [  6, 84, 10],
               [  9, 7, 64]])
assert S1.sum() == 300 and S2.sum() == 300

## 1. Accuracy with 95% confidence interval
Accuracy = (sum of diagonal) / n. Treat each review as a Bernoulli trial, so SE = sqrt(acc(1-acc)/n).
* **Wald:** acc +/- 1.96 SE (simple, but can misbehave near 0 or 1).
* **Wilson:** better coverage for moderate n.

In [2]:
def accuracy_ci(M, z=1.96):
    n = M.sum(); acc = np.trace(M) / n
    se = math.sqrt(acc * (1 - acc) / n)
    wald = (acc - z * se, acc + z * se)
    centre = (acc + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return acc, se, wald, (centre - half, centre + half)

acc, se, wald, wilson = accuracy_ci(S1)
print(f"Stage 1: acc={acc:.4f} ({np.trace(S1)}/{S1.sum()}), SE={se:.4f}")
print(f"  Wald   95% CI: [{wald[0]:.4f}, {wald[1]:.4f}]")
print(f"  Wilson 95% CI: [{wilson[0]:.4f}, {wilson[1]:.4f}]")

Stage 1: acc=0.7667 (230/300), SE=0.0244
  Wald   95% CI: [0.7188, 0.8145]
  Wilson 95% CI: [0.7156, 0.8110]


## 2. Per-class precision, recall, F1
For class c with rows = actual, columns = predicted:
* precision_c = M[c,c] / column total (of everything predicted c, how much was right)
* recall_c = M[c,c] / row total (of everything actually c, how much was found)
* F1_c = 2PR/(P+R)

In [3]:
def per_class(M):
    tp = np.diag(M).astype(float)
    P = tp / M.sum(axis=0); R = tp / M.sum(axis=1)
    F = 2 * P * R / (P + R)
    return pd.DataFrame({"TP": tp.astype(int), "Row total (support)": M.sum(axis=1),
                         "Col total": M.sum(axis=0), "Precision": P, "Recall": R, "F1": F},
                        index=labels)

pc1 = per_class(S1)
pc1.round(4)

,TP,Row total (support),Col total,Precision,Recall,F1
Positive,100,120,128,0.7812,0.8333,0.8065
Negative,80,100,95,0.8421,0.8000,0.8205
Neutral,50,80,77,0.6494,0.6250,0.6369


## 3. Macro, weighted and micro F1
* **Macro:** unweighted mean of per-class F1 (treats every class equally).
* **Weighted:** mean of per-class F1 weighted by support.
* **Micro:** pool all TP/FP/FN. For single-label multiclass, micro-P = micro-R = micro-F1 = accuracy.

In [4]:
def averages(M):
    pc = per_class(M)
    sup = pc["Row total (support)"].values
    macro = pc["F1"].mean()
    weighted = (pc["F1"] * sup).sum() / sup.sum()
    micro = np.trace(M) / M.sum()
    return macro, weighted, micro

m1 = averages(S1)
print("Stage 1  macro=%.4f  weighted=%.4f  micro=%.4f" % m1)

Stage 1  macro=0.7546  weighted=0.7659  micro=0.7667


## 4. Stage 1 error analysis
Off-diagonal cells show where errors are. Rows are actual, columns are predicted.

In [5]:
errs = []
for i in range(3):
    for j in range(3):
        if i != j:
            errs.append((labels[i] + " -> " + labels[j], S1[i, j]))
pd.DataFrame(sorted(errs, key=lambda t: -t[1]), columns=["Confusion (actual -> predicted)", "Count"])

,Confusion (actual -> predicted),Count
0,Neutral -> Positive,20
1,Positive -> Neutral,15
2,Negative -> Neutral,12
3,Neutral -> Negative,10
4,Negative -> Positive,8
5,Positive -> Negative,5


**Reading:** Neutral has the lowest precision and recall. 30 of 80 true Neutrals are misclassified (20 as Positive, 10 as Negative), and 27 Positive/Negative reviews are pushed into Neutral. Positive<->Neutral is the biggest confusion pair (20 + 15 = 35). The label name alone gives the LLM no definition of "Neutral", so mixed-sentiment reviews are handled inconsistently.

**Recommendations:** define Neutral explicitly, add few-shot examples including borderline mixed reviews, consider a short rubric or reasoning step, calibrate on confidence, and collect more borderline annotations.

## 5. Stage 2 re-evaluation and comparison

In [6]:
acc2, se2, wald2, wilson2 = accuracy_ci(S2)
print(f"Stage 2: acc={acc2:.4f} ({np.trace(S2)}/300), SE={se2:.4f}")
print(f"  Wald   95% CI: [{wald2[0]:.4f}, {wald2[1]:.4f}]")
print(f"  Wilson 95% CI: [{wilson2[0]:.4f}, {wilson2[1]:.4f}]")
pc2 = per_class(S2); display(pc2.round(4))
m2 = averages(S2)
print("Stage 2  macro=%.4f  weighted=%.4f  micro=%.4f" % m2)

cmp = pd.DataFrame({"Stage 1": [acc, *m1[:2]], "Stage 2": [acc2, *m2[:2]]},
                   index=["Accuracy / micro-F1", "Macro-F1", "Weighted-F1"])
cmp["Delta"] = cmp["Stage 2"] - cmp["Stage 1"]
display(cmp.round(4))
display((pc2[["Precision", "Recall", "F1"]] - pc1[["Precision", "Recall", "F1"]]).round(4).rename(columns=lambda c: "Delta " + c))

Stage 2: acc=0.8467 (254/300), SE=0.0208
  Wald   95% CI: [0.8059, 0.8874]
  Wilson 95% CI: [0.8015, 0.8830]


,TP,Row total (support),Col total,Precision,Recall,F1
Positive,106,120,121,0.8760,0.8833,0.8797
Negative,84,100,95,0.8842,0.8400,0.8615
Neutral,64,80,84,0.7619,0.8000,0.7805


Stage 2  macro=0.8406  weighted=0.8472  micro=0.8467


,Stage 1,Stage 2,Delta
Accuracy / micro-F1,0.7667,0.8467,0.0800
Macro-F1,0.7546,0.8406,0.0859
Weighted-F1,0.7659,0.8472,0.0812


,Delta Precision,Delta Recall,Delta F1
Positive,0.0948,0.050,0.0732
Negative,0.0421,0.040,0.0410
Neutral,0.1126,0.175,0.1435


**Reading:** accuracy rises by about 8 points and macro-F1 by about 8.6 points. Neutral improves most (F1 0.637 -> 0.780). The Wald CIs overlap slightly, but both stages are scored on the *same* 300 reviews, so a paired test (McNemar, using per-review correctness) is the right significance check. It is not possible here because only aggregate confusion matrices are given.

## 6. Prompt stability over 5 example orderings
Report mean and **sample** standard deviation (ddof = 1). Population SD is shown for reference.

In [7]:
runs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
print(f"mean = {runs.mean():.4f}")
print(f"sample SD (ddof=1) = {runs.std(ddof=1):.4f}   population SD = {runs.std(ddof=0):.4f}")
print(f"range = [{runs.min():.3f}, {runs.max():.3f}]  spread = {runs.max()-runs.min():.3f}")
print(f"Stage1->Stage2 gain = {acc2-acc:.3f}  (about {(acc2-acc)/runs.std(ddof=1):.0f}x the ordering SD)")

mean = 0.8454
sample SD (ddof=1) = 0.0067   population SD = 0.0060
range = [0.837, 0.853]  spread = 0.016
Stage1->Stage2 gain = 0.080  (about 12x the ordering SD)


**Reading:** ordering changes accuracy by under 2 points (SD about 0.007), far smaller than the Stage 1 -> Stage 2 gain, so the improvement is not an ordering artefact. n = 5 is small, so treat the SD as a rough estimate.

## 7. Generated-text metrics (implemented from scratch)
Reference r = *the delivery was late but the product quality is good* (10 tokens)
Candidate c = *the product quality is good but delivery was slow* (9 tokens)

In [8]:
ref = "the delivery was late but the product quality is good".split()
cand = "the product quality is good but delivery was slow".split()

def ngrams(t, n): return Counter(tuple(t[i:i+n]) for i in range(len(t) - n + 1))
def clipped(c, r, n):
    cc, rr = ngrams(c, n), ngrams(r, n)
    return sum(min(v, rr[k]) for k, v in cc.items()), sum(cc.values()), sum(rr.values())

# BLEU-4
ps = []
for n in range(1, 5):
    m, tc, _ = clipped(cand, ref, n); ps.append(m / tc)
    print(f"p{n} = {m}/{tc} = {m/tc:.4f}")
bp = 1.0 if len(cand) > len(ref) else math.exp(1 - len(ref) / len(cand))
bleu4 = bp * math.exp(np.mean(np.log(ps)))
print(f"BP = exp(1 - {len(ref)}/{len(cand)}) = {bp:.4f}  ->  BLEU-4 = {bleu4:.4f}")

# ROUGE-N
def rouge_n(c, r, n):
    m, tc, tr = clipped(c, r, n); P, R = m / tc, m / tr
    return P, R, (2 * P * R / (P + R) if m else 0.0)
r1, r2 = rouge_n(cand, ref, 1), rouge_n(cand, ref, 2)

# ROUGE-L via LCS
def lcs(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i+1][j+1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j+1], dp[i+1][j])
    return dp[-1][-1]
L = lcs(cand, ref); PL, RL = L / len(cand), L / len(ref); FL = 2*PL*RL/(PL+RL)

# METEOR (exact matching only), alpha=0.9, beta=3, gamma=0.5
# Hand-derived minimal-chunk alignment (the -> second 'the' of the reference):
align = {0: 5, 1: 6, 2: 7, 3: 8, 4: 9, 5: 4, 6: 1, 7: 2}   # cand idx -> ref idx
for ci, ri in align.items(): assert cand[ci] == ref[ri]
m = len(align); P, R = m / len(cand), m / len(ref)
fmean = P * R / (0.9 * P + 0.1 * R)           # = 10PR/(R+9P): recall weighted 9x precision
pairs = sorted(align.items())
chunks = 1 + sum(1 for (c0, r0), (c1, r1_) in zip(pairs, pairs[1:]) if not (c1 == c0 + 1 and r1_ == r0 + 1))
penalty = 0.5 * (chunks / m) ** 3
meteor = fmean * (1 - penalty)

res = pd.DataFrame({"P": [r1[0], r2[0], PL], "R": [r1[1], r2[1], RL], "F1": [r1[2], r2[2], FL]},
                   index=["ROUGE-1", "ROUGE-2", "ROUGE-L"])
display(res.round(4))
print(f"LCS length = {L}")
print(f"METEOR: m={m}, P={P:.4f}, R={R:.4f}, Fmean={fmean:.4f}, chunks={chunks}, penalty={penalty:.4f}, METEOR={meteor:.4f}")

p1 = 8/9 = 0.8889
p2 = 5/8 = 0.6250
p3 = 3/7 = 0.4286
p4 = 2/6 = 0.3333
BP = exp(1 - 10/9) = 0.8948  ->  BLEU-4 = 0.4750


,P,R,F1
ROUGE-1,0.8889,0.8000,0.8421
ROUGE-2,0.6250,0.5556,0.5882
ROUGE-L,0.5556,0.5000,0.5263


LCS length = 5
METEOR: m=8, P=0.8889, R=0.8000, Fmean=0.8081, chunks=3, penalty=0.0264, METEOR=0.7868


**Optional cross-check with NLTK.** NLTK's METEOR also uses stemming and WordNet synonyms, and its tie-breaking may differ slightly. State your settings in the report. If your environment has no internet access, skip this cell.

In [9]:
try:
    import nltk
    from nltk.translate.meteor_score import meteor_score
    from nltk.translate.bleu_score import sentence_bleu
    nltk.download("wordnet", quiet=True)
    print("NLTK METEOR :", round(meteor_score([ref], cand), 4))
    print("NLTK BLEU-4 :", round(sentence_bleu([ref], cand, weights=(.25,)*4), 4), "(no smoothing)")
except Exception as e:
    print("NLTK cross-check skipped:", e)

NLTK METEOR : 0.7868
NLTK BLEU-4 : 0.475 (no smoothing)


**Reading:** "late" vs "slow" is a paraphrase, and the clause order is swapped. Unigram metrics (ROUGE-1 0.842) credit the shared vocabulary, while higher-order n-gram metrics (BLEU-4 0.475, ROUGE-2 0.588, ROUGE-L 0.526) are lower because the reordering breaks long matches. METEOR is relatively high because its fragmentation penalty is mild (3 chunks).

## 8. Perplexity of the generated justification
PPL = exp( -(1/N) * sum ln p_i ) over the N = 9 generated tokens.

In [10]:
p = np.array([0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20])
logs = np.log(p)
print("ln p:", np.round(logs, 4))
print(f"sum ln p = {logs.sum():.4f}, mean = {logs.mean():.4f}")
ppl = math.exp(-logs.mean())
print(f"Perplexity = {ppl:.4f}")

ln p: [-0.5108 -0.9163 -0.6931 -0.2231 -0.3567 -1.204  -1.3863 -0.6931 -1.6094]
sum ln p = -7.5929, mean = -0.8437
Perplexity = 2.3249


**Reading:** PPL about 2.32 means the model was, on average, as uncertain as choosing among about 2.3 equally likely tokens per step. That is low (confident), though uneven: tokens at 0.20, 0.25 and 0.30 were the weak points. Perplexity measures the model's own confidence, not faithfulness to the reference or correctness.